In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

/home/joaquin/Documents/GitHub/skforecast
0.22.0


Benchmark diseñado para comparar el rendimiento de dos estrategias de
extracción de series temporales individuales desde un DataFrame con
MultiIndex (series_id, datetime).

El conjunto de prueba contiene 1000 series, cada una con 1000 valores
horarios, para un total de 1.000.000 de filas.

Se comparan:
1) acceso iterativo mediante .loc por cada identificador de serie
2) particionado mediante groupby(level=0)

Además del tiempo de ejecución, se verifica de forma robusta que las
series resultantes preservan tanto la metadata de frecuencia (.freq)
como la regularidad temporal real mediante pd.infer_freq().

El objetivo es evaluar simultáneamente rendimiento, escalabilidad y
preservación de la estructura temporal.

In [2]:
import pandas as pd
import numpy as np
import timeit
import statistics

# ==========================================
# Configuración del dataset
# ==========================================
n_series = 1000
n_values = 1000
expected_freq = "h"
n_runs = 5

# ==========================================
# Crear dataset de prueba
# ==========================================
dates = pd.date_range(
    start="2024-01-01",
    periods=n_values,
    freq=expected_freq
)

series_ids = [f"serie_{i}" for i in range(n_series)]

index = pd.MultiIndex.from_product(
    [series_ids, dates],
    names=["series_id", "datetime"]
)

df = pd.DataFrame(
    {
        "value": np.random.rand(n_series * n_values)
    },
    index=index
)

first_col = "value"

print(f"DataFrame creado: {len(df):,} filas")
print(f"Número de series: {n_series}")
print(f"Valores por serie: {n_values}")
print()

# ==========================================
# Método 1: usando loc
# ==========================================
def method_loc():
    return {
        sid: df.loc[sid][first_col].rename(sid)
        for sid in df.index.get_level_values(0).unique()
    }

# ==========================================
# Método 2: usando groupby
# ==========================================
def method_groupby():
    return {
        sid: group[first_col].droplevel(0).rename(sid)
        for sid, group in df.groupby(level=0, sort=True)
    }

# ==========================================
# Verificación robusta de frecuencia
# ==========================================
def check_frequency_robust(series_dict, expected_freq="h"):
    lost_metadata = []
    irregular = []

    for sid, s in series_dict.items():
        idx = s.index

        # Verifica si se pierde metadata freq
        if idx.freq is None:
            lost_metadata.append(sid)

        # Verifica regularidad real
        inferred = pd.infer_freq(idx)

        if inferred is None:
            irregular.append((sid, "No inferible"))
        elif inferred.lower() != expected_freq.lower():
            irregular.append((sid, inferred))

    return {
        "lost_metadata_count": len(lost_metadata),
        "irregular_count": len(irregular),
        "lost_metadata_examples": lost_metadata[:5],
        "irregular_examples": irregular[:5],
    }

# ==========================================
# Benchmark
# ==========================================
print("Ejecutando benchmark...")

loc_times = timeit.repeat(method_loc, repeat=n_runs, number=1)
grp_times = timeit.repeat(method_groupby, repeat=n_runs, number=1)

loc_avg = statistics.mean(loc_times)
grp_avg = statistics.mean(grp_times)

print("\n=== RESULTADOS DE VELOCIDAD ===")
print(f"LOC       -> promedio: {loc_avg:.4f} s")
print(f"GROUPBY   -> promedio: {grp_avg:.4f} s")
print(f"Speedup   -> {loc_avg / grp_avg:.2f}x")
print()

# ==========================================
# Verificación de frecuencia
# ==========================================
print("Verificando frecuencia...")

loc_result = method_loc()
grp_result = method_groupby()

loc_check = check_frequency_robust(loc_result, expected_freq)
grp_check = check_frequency_robust(grp_result, expected_freq)

print("\n=== VERIFICACIÓN ROBUSTA DE FRECUENCIA ===")

print("\nMétodo LOC")
print(f"Series con metadata freq perdida: {loc_check['lost_metadata_count']}")
print(f"Series irregulares reales: {loc_check['irregular_count']}")
print(f"Ejemplos metadata perdida: {loc_check['lost_metadata_examples']}")
print(f"Ejemplos irregulares: {loc_check['irregular_examples']}")

print("\nMétodo GROUPBY")
print(f"Series con metadata freq perdida: {grp_check['lost_metadata_count']}")
print(f"Series irregulares reales: {grp_check['irregular_count']}")
print(f"Ejemplos metadata perdida: {grp_check['lost_metadata_examples']}")
print(f"Ejemplos irregulares: {grp_check['irregular_examples']}")

DataFrame creado: 1,000,000 filas
Número de series: 1000
Valores por serie: 1000

Ejecutando benchmark...

=== RESULTADOS DE VELOCIDAD ===
LOC       -> promedio: 1.4709 s
GROUPBY   -> promedio: 0.3368 s
Speedup   -> 4.37x

Verificando frecuencia...

=== VERIFICACIÓN ROBUSTA DE FRECUENCIA ===

Método LOC
Series con metadata freq perdida: 0
Series irregulares reales: 0
Ejemplos metadata perdida: []
Ejemplos irregulares: []

Método GROUPBY
Series con metadata freq perdida: 0
Series irregulares reales: 0
Ejemplos metadata perdida: []
Ejemplos irregulares: []
